# 02: Molecular Generation with MolGPT

In this notebook, we explore **MolGPT**, a generative model based on the GPT-2 architecture that has been trained to "speak" the language of chemistry by learning SMILES strings.

## 1. What is MolGPT?

MolGPT is a Transformer-based model (GPT-2) trained on 1.5 million SMILES strings from the ZINC database. Instead of predicting the next word in a sentence, it predicts the next character in a SMILES string. By doing so, it learns the grammar of valid chemical structures.

> **Note:** In production, generation is often *conditioned* on a specific target or property. Here, we are performing *unconditional* generation to understand the model's baseline capabilities first.

## 2. Load the model

We will use the `transformers` library from HuggingFace to load a pre-trained MolGPT model.

In [ ]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer
from rdkit import Chem
from rdkit.Chem import Draw, Descriptors
import torch

model_name = "entropy/gpt2_zinc_87m"
print(f"Loading model: {model_name}...")

tokenizer = GPT2Tokenizer.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name)

print("Model loaded successfully!")

## 3. Generate raw SMILES

Let's ask the model to generate 10 new molecular structures. We use `temperature=0.8` to add some variety to the generation.

In [ ]:
def generate_smiles(num_samples=10):
    # Start with an empty string or a beginning-of-SMILES token if applicable
    input_ids = tokenizer.encode("", return_tensors="pt")
    
    outputs = model.generate(
        input_ids,
        do_sample=True,
        max_length=100,
        temperature=0.8,
        top_k=50,
        num_return_sequences=num_samples,
        pad_token_id=tokenizer.eos_token_id
    )
    
    decoded = [tokenizer.decode(output, skip_special_tokens=True).strip() for output in outputs]
    return decoded

raw_outputs = generate_smiles(10)
print("Raw Generated SMILES:")
for i, s in enumerate(raw_outputs):
    print(f"{i+1}: {s}")

## 4. Filter valid molecules

Not every string generated by the model will be a valid molecule. We use RDKit to filter out the mistakes.

In [ ]:
def is_valid_smiles(smiles: str) -> bool:
    return Chem.MolFromSmiles(smiles) is not None

valid_smiles = [s for s in raw_outputs if is_valid_smiles(s)]
valid_mols = [Chem.MolFromSmiles(s) for s in valid_smiles]

print(f"Generated: {len(raw_outputs)}")
print(f"Valid: {len(valid_smiles)}")
print(f"Validity Rate: {len(valid_smiles)/len(raw_outputs)*100:.1f}%")

## 5. Visualise the valid ones

Let's see what the model "dreamed" up!

In [ ]:
if valid_mols:
    img = Draw.MolsToGridImage(valid_mols, molsPerRow=5, subImgSize=(200, 200), legends=valid_smiles)
    display(img)
else:
    print("No valid molecules were generated in this run.")

## 6. Score them

We can evaluate how "drug-like" these generated molecules are using Lipinski's Rule of 5.

In [ ]:
import pandas as pd

def get_scores(smiles_list):
    results = []
    for s in smiles_list:
        mol = Chem.MolFromSmiles(s)
        res = {
            "SMILES": s,
            "MW": Descriptors.MolWt(mol),
            "LogP": Descriptors.MolLogP(mol),
            "HBD": Descriptors.NumHDonors(mol),
            "HBA": Descriptors.NumHAcceptors(mol)
        }
        # Ro5 criteria
        res["Ro5_Pass"] = (res["MW"] < 500 and res["LogP"] < 5 and 
                             res["HBD"] < 5 and res["HBA"] < 10)
        results.append(res)
    return pd.DataFrame(results)

if valid_smiles:
    df = get_scores(valid_smiles)
    display(df)
else:
    print("Nothing to score.")

## 7. Observations

Take a look at the generated grid and the table above. 

- **Diversity:** Are the molecules similar to each other, or quite different?
- **Complexity:** Does the model generate simple chains or complex ring systems?
- **Druglikeness:** Based on the Lipinski scores, which of these "dreams" could potentially be turned into a real medicine?

Molecular generation is an iterative process. By tweaking the model and the generation parameters (like temperature), we can explore different regions of the vast chemical space.